#### DATASET 'sinistros'
- nome_segurado, str
- tipo_sinistro, str
- nome_beneficiario, str
- valor_sinistro, float
- quem_forma_beneficiados, str
- status_seguro, str
- regiao_sinistro, str

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, when, count, mean, stddev, to_date, lit, trim, round
from pyspark.sql.types import IntegerType, DoubleType, DateType, LongType, StringType
from functools import reduce

caminho = "/Workspace/Users/actc@cesar.school/Grupo7-Setor-de-Seguros/data/raw/sinistros.csv"

df = spark.read\
    .format("csv")\
    .option("header", "true")\
    .option("sep", ",")\
    .load(caminho)

print(df.columns)
df.printSchema()
df.show()

### Limpeza inicial
- casting, trim, duplicatas, valores anormais, nulos

In [0]:

df = df.withColumn(
    "nome_segurado",
    trim(
        regexp_replace(
            col("nome_segurado"),
            "^(Dr\\.|Dra\\.|Sr\\.|Sra\\.|Srta\\.)\\s+", ""
        )
    )
).withColumn(
    "nome_beneficiario",
    trim(
        regexp_replace(
            col("nome_beneficiario"),
            "^(Dr\\.|Sr\\.|Sra\\.|Srta\\.)\\s+", ""
        )
    )
).withColumn(
    "quem_forma_beneficiados",
    trim(
        regexp_replace(
            col("quem_forma_beneficiados"),
            "^(Dr\\.|Sr\\.|Sra\\.|Srta\\.)\\s+", ""
        )
    )
)

In [0]:
for c in df.columns:
    df = df.withColumn(c, trim(regexp_replace(col(c), " +", " ")))

df = df.withColumn("valor_sinistro", regexp_replace(col("valor_sinistro"), ",", ".").cast(DoubleType()))

Checamos se há valores monetários negativos

In [0]:
negative_rows = df.filter(col('valor_sinistro') < 0)
negative_count = negative_rows.count()

print(f"Encontrou {negative_count} linhas com valores negativos.") # Não encontramos
if negative_count > 0:
    display(negative_rows)

Checamos se há valores discrepantes (+- 4 desvios padrão da média) e removemos

In [0]:

stats = df.select(mean(col("valor_sinistro")).alias("avg"), stddev(col("valor_sinistro")).alias("std")).collect()[0]
mean_val = stats["avg"]
std_val = stats["std"]
    
if mean_val is not None and std_val is not None:
    lower_bound = max(0, mean_val - (4 * std_val))
    upper_bound = mean_val + (4 * std_val)
    
    outliers = df.filter((col("valor_sinistro") < lower_bound) | (col("valor_sinistro") > upper_bound))
    count_outliers = outliers.count()
    
    if count_outliers > 0:
        print(f"Coluna valor_sinistr': {count_outliers} outliers (De {lower_bound:.2f} até {upper_bound:.2f})")
        display(outliers)
        df = df.filter((col("valor_sinistro") >= lower_bound) & (col("valor_sinistro") <= upper_bound))
else:
    print("Erro ao calcular medidas.")

Checa se há linhas duplicadas

In [0]:
total_count = df.count()
distinct_count = df.distinct().count()
duplicate_count = total_count - distinct_count

print(f"Total: {total_count}")
print(f"Distintas: {distinct_count}")
print(f"Duplicatas: {duplicate_count}")

if duplicate_count > 0:
    df.groupby(df.columns).count().filter("count > 1").show()

Checamos se há dados nulos ou vazios

In [0]:
null_check_exprs = []
for c in df.columns:
    if isinstance(df.schema[c].dataType, StringType):
        null_check_exprs.append(count(when(col(c).isNull() | (col(c) == ""), c)).alias(c))
    else:
        null_check_exprs.append(count(when(col(c).isNull(), c)).alias(c))

null_counts = df.select(null_check_exprs)
display(null_counts)

In [0]:
# Transforma strings vazias em NULL e remove linhas
for c in df.columns:
    if isinstance(df.schema[c].dataType, StringType):
        df = df.withColumn(c, when(col(c) == "", None).otherwise(col(c)))

df = df.dropna(how='any')

Removemos pessoas que morreram mais de uma vez

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

def remove_suspicious_death_claims(df):
    w = Window.partitionBy("nome_segurado")
    
    is_death_claim = F.when(col("tipo_sinistro") == "Morte Acidental", 1).otherwise(0)
    
    df = df.withColumn("cnt_morte", F.sum(is_death_claim).over(w))
    
    df_filtered = df.filter(
        ~( (F.col("tipo_sinistro") == "Morte Acidental") & (F.col("cnt_morte") > 1) )
    )

    df_final = df_filtered.drop("cnt_morte")
    
    return df_final

df = remove_suspicious_death_claims(df)

In [0]:
display(df)